# 파이프라인 — 셀 두 개로 제출까지

원본 csv 에서 fold · 피처 파켓 · seed 앙상블 학습 · 가중 블렌드 + 로짓 보정 · 짝 라벨 규칙 ·
제출 파일 · 실행 로그까지 한 번에 만든다.

**로직은 이 노트북에 없다.** 전부 `cancer_hack.pipeline` 에 있고, `scripts/run_pipeline.py`
가 부르는 것과 **같은 함수**다. 진입점이 둘이어도 코드 경로가 하나여야 결과가 안 어긋난다.

```
notebooks/13_pipeline.ipynb  ─┐
                              ├─→ cancer_hack.pipeline.run_pipeline()
scripts/run_pipeline.py      ─┘
```

## 무엇이 고정돼 있나

| 축 | 값 | 어디 |
|---|---|---|
| seed 앙상블 | 42 · 7 · 2024 | `cancer_hack.validation.SEED_ENSEMBLE` |
| fold 분할 seed | 42 | `cancer_hack.validation.SEED` |
| 잠재 SVD/NMF · KMeans 모듈 | 0 | `train_gbdt` 기본값 |
| `gpu_ram_part` | **0.4 (숫자 고정)** | `PipelineConfig` |
| 피처셋 | `f16` 16블록 5,309열 | `PipelineConfig` |
| 결합 가중 | 0.45 / 0.45 / 0.10 ÷ seed 수 | `PipelineConfig` |
| 짝 규칙 | 항상 켬 · `min_mut=3` | `PipelineConfig` |

`gpu_ram_part` 가 왜 숫자여야 하는지가 중요하다. `auto` 는 **실행 시점의 GPU 여유 메모리**로
값을 정해서, 같은 seed 로도 CatBoost 가 6,201행 중 56행의 라벨을 다르게 낸다. 숫자로 박으면
확률까지 비트 단위로 같아진다. 실측은 `docs/reproducibility.md`.

## 출력

기존 `data/process/`·`artifacts/` 는 건드리지 않는다. 실행별로 갈린다.

```
data/process_<RUN_TAG>/      피처 파켓 26개
artifacts/runs/<RUN_TAG>/    oof · test_predictions · logs · submissions
  ├── run.json               설정·환경·파켓 지문·단계별 시간·점수 전부
  └── run.md                 사람이 읽는 요약
```

## 걸리는 시간

GPU 기준 피처 16분 + 학습 30분(모델 3 × seed 3) ≈ 45분. 피처를 한 번 만든 뒤에는
`rebuild_features=False` 로 두고 모델만 다시 돌릴 수 있다.

**DACON 업로드는 사람이 직접 한다.** 이 노트북은 로컬 csv 만 만든다.

## 1. 설정

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from cancer_hack.pipeline import PipelineConfig
from cancer_hack.validation import SEED, SEED_ENSEMBLE

config = PipelineConfig(
    run_tag="nb13_seed3",
    seeds=SEED_ENSEMBLE,          # 42 · 7 · 2024 — 한 곳에서 온다
    feature_config="f16",
    cv="sgkf",                    # fold_group5
    model_weights={"xgb": 0.45, "catboost": 0.45, "rf": 0.10},
    min_mut=3,
    apply_pair_rule=True,
    gpu_ram_part=0.4,             # 숫자 고정 — auto 는 재현을 깬다
    rebuild_features=True,        # 한 번 만들었으면 False
    reference=ROOT.parent / "Models/pairrule_candidates/submission_ens16_seed3_pairrule_m3.csv",
)

for key, value in config.as_dict().items():
    print(f"  {key:18s} {value}")
print(f"\n소스 {len(config.models)}모델 × {len(config.seeds)}seed = {len(config.source_weights)}개")
print(f"가중 {[round(w, 4) for w in config.source_weights]}  합 {sum(config.source_weights):.6f}")

## 2. 실행

한 줄이다. 단계마다 시작·완료와 소요 시간이 찍히고, 실패하면 그 단계가 `run.json` 에
`실패` 로 남고 예외가 그대로 올라온다 — 실패한 실행이 성공처럼 보이는 게 최악이라서다.

In [ ]:
from cancer_hack.pipeline import run_pipeline

result = run_pipeline(config)

## 3. 결과

In [ ]:
import pandas as pd

log = result["log"]
print(log.summary())
print(f"\nOOF macro F1 = {result['oof_macro_f1']:.4f}")

print("\n소스별 OOF")
for key, value in log.values.items():
    if key.startswith("oof_") and key not in ("oof_uniform", "oof_blend_raw", "oof_blend_calibrated"):
        print(f"  {key[4:]:20s} {value:.4f}")

print(f"\n균등 가중        {log.values['oof_uniform']:.4f}")
print(f"고정 가중 raw    {log.values['oof_blend_raw']:.4f}")
print(f"고정 가중 + 보정 {log.values['oof_blend_calibrated']:.4f}   ← 보고할 값")

print(f"\n짝 규칙 대상 {log.values['pair_rule_rows']}행 · 바뀐 행 {log.values['pair_rule_changed']}")
if "reference_rows_differ" in log.values:
    n = log.values["reference_rows_differ"]
    print(f"기준 파일과 다른 행 {n}" + ("  — 재현 성공" if n == 0 else ""))

print()
print(pd.DataFrame(
    [(s["name"], s.get("status"), s.get("seconds")) for s in log.steps],
    columns=["단계", "상태", "초"],
).to_string(index=False))

### 만들어진 파일

In [ ]:
for label, path in log.artifacts.items():
    print(f"  {label:22s} {path}")
for label, path in result["run_files"].items():
    print(f"  {label:22s} {path}")

print("\n※ DACON 업로드는 사람이 직접 한다.")

## 4. 같은 것을 CLI 로

노트북을 못 쓰는 환경(Colab·Kaggle·CI)에서는 이 명령이 정확히 같은 일을 한다.

```bash
python scripts/run_pipeline.py --run-tag nb13_seed3 \
    --reference ../Models/pairrule_candidates/submission_ens16_seed3_pairrule_m3.csv
```

기본값이 위 설정과 같으므로 인자를 안 줘도 된다. 바꾸고 싶은 축만 준다.

```bash
python scripts/run_pipeline.py --run-tag quick --seeds 42 --no-rebuild-features
python scripts/run_pipeline.py --run-tag cbopt --catboost-params cbopt10
python scripts/run_pipeline.py --run-tag cpu --device cpu        # 완전 재현이 필요할 때
```

## 5. 재검증

이 노트북은 새 파켓을 만든다. **기존 제출본을 다시 확인할 때는** 기본 경로로 돌아가
`09`·`10`·`12` 노트북을 쓴다.

```python
from cancer_hack.paths import reset_run_dirs
reset_run_dirs()
```

기존 `data/process/mutation_encoded.parquet` 은 `encode_mutation` 정정 전 코드로 만들어져
지금 만드는 것과 68셀이 다르다. **두 쪽 OOF 를 한 블렌드에 넣으면 안 된다.**